# TCREmp VDJdb Analysis (mirpy)

This notebook reproduces the core `tcremp_vdjdb` workflow using `mirpy` APIs and
AIRR benchmark assets from Hugging Face.

Pipeline:
1. Load VDJdb slim from `isalgo/airr_benchmark`.
2. Build TRB clonotypes and labels (`antigen.epitope`).
3. Embed with `TCREmp`.
4. Evaluate representation with PCA/UMAP, DBSCAN, and classifiers.
5. Compare `fixed_gap` vs `biopython` backends (runtime + predictive quality).

In [ ]:
# Setup imports, deterministic seed, and package versions for reproducibility.
import os
import sys
import time
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.decomposition import PCA
from sklearn.cluster import DBSCAN
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    f1_score,
    adjusted_rand_score,
    normalized_mutual_info_score,
    silhouette_score,
    classification_report,
)

try:
    import umap
except ImportError as exc:
    raise ImportError("Please install umap-learn in the notebook kernel: %pip install umap-learn") from exc

repo_root = Path.cwd().resolve().parent if Path.cwd().name == "notebooks" else Path.cwd().resolve()
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

from mir.common.clonotype import Clonotype
from mir.embedding.tcremp import TCREmp
from mir.utils.notebook_assets import ensure_airr_benchmark, find_airr_benchmark_vdjdb_slim

SEED = 42
np.random.seed(SEED)

print(f"Python: {sys.version.split()[0]}")
print(f"numpy: {np.__version__}")
print(f"pandas: {pd.__version__}")

In [ ]:
# Download/load VDJdb slim from AIRR benchmark assets and show schema.
benchmark_root = ensure_airr_benchmark(repo_root, allow_patterns=["vdjdb/**"])
vdjdb_path = find_airr_benchmark_vdjdb_slim(benchmark_root)
vdjdb = pd.read_csv(vdjdb_path, sep='\t')

print(f"Loaded: {vdjdb_path}")
print(f"Rows: {len(vdjdb):,}  Columns: {len(vdjdb.columns)}")
print(vdjdb.columns.tolist()[:20])

In [ ]:
# Prepare TRB epitope dataset with robust filtering and class balancing limits.
label_col = "antigen.epitope"
required_cols = ["gene", "cdr3", "v.segm", "j.segm", label_col]
missing_cols = [c for c in required_cols if c not in vdjdb.columns]
if missing_cols:
    raise ValueError(f"Missing required columns in VDJdb slim: {missing_cols}")

df = vdjdb.copy()
if "species" in df.columns:
    df = df[df["species"].astype(str).str.lower().isin(["homosapiens", "human"])].copy()
df = df[df["gene"].eq("TRB")].copy()
df = df.dropna(subset=["cdr3", "v.segm", "j.segm", label_col]).copy()

for c in ["cdr3", "v.segm", "j.segm", label_col]:
    df[c] = df[c].astype(str).str.strip()

df = df[(df["cdr3"].str.len() > 0) & (df["v.segm"].str.len() > 0) & (df["j.segm"].str.len() > 0)]
df = df.drop_duplicates(subset=["cdr3", "v.segm", "j.segm", label_col]).copy()

min_per_epitope = 50
max_per_epitope = 400
top_epitopes = 25

counts = df[label_col].value_counts()
eligible = counts[counts >= min_per_epitope].head(top_epitopes).index
df = df[df[label_col].isin(eligible)].copy()

parts = []
for ep, grp in df.groupby(label_col):
    n_take = min(len(grp), max_per_epitope)
    parts.append(grp.sample(n=n_take, random_state=SEED))
df = pd.concat(parts, ignore_index=True)

print(f"Prepared rows: {len(df):,}")
print(f"Epitopes: {df[label_col].nunique()}")
display(df[label_col].value_counts().head(20))

In [ ]:
# Convert rows to mirpy Clonotype objects for TCREmp embedding.
clonotypes = [
    Clonotype(
        sequence_id=str(i),
        locus="TRB",
        v_gene=row["v.segm"],
        j_gene=row["j.segm"],
        junction_aa=row["cdr3"],
        duplicate_count=1,
        _validate=False,
    )
    for i, row in df.reset_index(drop=True).iterrows()
]
labels = df[label_col].to_numpy()

print(f"Clonotypes: {len(clonotypes):,}")
print(f"Unique labels: {len(set(labels))}")

In [ ]:
# Build TCREmp model and compute fixed-gap embeddings (default production backend).
n_prototypes = 1000
model_fixed = TCREmp.from_defaults(
    species="human",
    locus="TRB",
    n_prototypes=n_prototypes,
    junction_method="fixed_gap",
)

t0 = time.perf_counter()
X_fixed = model_fixed.embed(clonotypes, n_jobs=None)
t_fixed = time.perf_counter() - t0

pairs = len(clonotypes) * n_prototypes
print(f"Embedding shape: {X_fixed.shape}")
print(f"Runtime (fixed-gap): {t_fixed:.2f}s")
print(f"Effective pairs/s: {pairs / max(t_fixed, 1e-9):,.0f}")

In [ ]:
# Compute PCA/UMAP projections for visualization and downstream clustering.
le = LabelEncoder()
y = le.fit_transform(labels)

pca = PCA(n_components=50, random_state=SEED)
X_pca = pca.fit_transform(X_fixed)

umap_model = umap.UMAP(
    n_components=2,
    n_neighbors=30,
    min_dist=0.1,
    metric="euclidean",
    random_state=SEED,
)
X_umap = umap_model.fit_transform(X_pca[:, :30])

plot_df = pd.DataFrame({
    "label": labels,
    "pca1": X_pca[:, 0],
    "pca2": X_pca[:, 1],
    "umap1": X_umap[:, 0],
    "umap2": X_umap[:, 1],
})

print("PCA explained variance (first 5):", np.round(pca.explained_variance_ratio_[:5], 4))

In [ ]:
# Plot PCA and UMAP colored by epitope (top labels only for readability).
top_labels = plot_df["label"].value_counts().head(12).index
viz = plot_df[plot_df["label"].isin(top_labels)].copy()

fig, axes = plt.subplots(1, 2, figsize=(15, 6))
sns.scatterplot(data=viz, x="pca1", y="pca2", hue="label", s=18, alpha=0.75, ax=axes[0], legend=False)
axes[0].set_title("TCREmp PCA (top 12 epitopes)")

sns.scatterplot(data=viz, x="umap1", y="umap2", hue="label", s=18, alpha=0.75, ax=axes[1])
axes[1].set_title("TCREmp UMAP (top 12 epitopes)")
axes[1].legend(loc="center left", bbox_to_anchor=(1.02, 0.5), frameon=False)

plt.tight_layout()
plt.show()

In [ ]:
# Run DBSCAN on reduced embeddings and report clustering quality metrics.
X_cluster = StandardScaler().fit_transform(X_pca[:, :20])
dbscan = DBSCAN(eps=1.8, min_samples=10, metric="euclidean")
cluster_labels = dbscan.fit_predict(X_cluster)

mask_core = cluster_labels != -1
ari = adjusted_rand_score(y, cluster_labels)
nmi = normalized_mutual_info_score(y, cluster_labels)
sil = silhouette_score(X_cluster[mask_core], cluster_labels[mask_core]) if mask_core.sum() > 10 and len(np.unique(cluster_labels[mask_core])) > 1 else np.nan

print(f"DBSCAN clusters (excluding noise): {len(set(cluster_labels)) - (1 if -1 in cluster_labels else 0)}")
print(f"Noise fraction: {(cluster_labels == -1).mean():.3f}")
print(f"ARI: {ari:.4f}")
print(f"NMI: {nmi:.4f}")
print(f"Silhouette (non-noise): {sil:.4f}")

In [ ]:
# Train/test classifiers on TCREmp embeddings and report predictive performance.
X_train, X_test, y_train, y_test = train_test_split(
    X_pca[:, :30],
    y,
    test_size=0.3,
    random_state=SEED,
    stratify=y,
)

rf = RandomForestClassifier(n_estimators=500, random_state=SEED, n_jobs=-1)
rf.fit(X_train, y_train)
rf_pred = rf.predict(X_test)

lr = LogisticRegression(max_iter=3000, random_state=SEED, multi_class="multinomial")
lr.fit(X_train, y_train)
lr_pred = lr.predict(X_test)

metrics_df = pd.DataFrame([
    {
        "model": "RandomForest",
        "accuracy": accuracy_score(y_test, rf_pred),
        "macro_f1": f1_score(y_test, rf_pred, average="macro"),
    },
    {
        "model": "LogisticRegression",
        "accuracy": accuracy_score(y_test, lr_pred),
        "macro_f1": f1_score(y_test, lr_pred, average="macro"),
    },
])
display(metrics_df.sort_values("macro_f1", ascending=False))

print("\nRandomForest detailed report:\n")
print(classification_report(y_test, rf_pred, target_names=le.classes_, zero_division=0))

In [ ]:
# Compare fixed-gap vs BioPython on a controlled subset (runtime + classifier quality).
compare_n = min(2000, len(clonotypes))
cmp_idx = np.random.RandomState(SEED).choice(len(clonotypes), size=compare_n, replace=False)
cmp_clonotypes = [clonotypes[i] for i in cmp_idx]
cmp_labels = y[cmp_idx]

model_bio = TCREmp.from_defaults(
    species="human",
    locus="TRB",
    n_prototypes=500,
    junction_method="biopython",
)
model_fg_cmp = TCREmp.from_defaults(
    species="human",
    locus="TRB",
    n_prototypes=500,
    junction_method="fixed_gap",
)

t0 = time.perf_counter()
X_fg_cmp = model_fg_cmp.embed(cmp_clonotypes, n_jobs=None)
t_fg_cmp = time.perf_counter() - t0

t0 = time.perf_counter()
X_bio_cmp = model_bio.embed(cmp_clonotypes, n_jobs=None)
t_bio_cmp = time.perf_counter() - t0

def eval_rf(X, y_vec):
    Xp = PCA(n_components=min(30, X.shape[1]), random_state=SEED).fit_transform(X)
    Xtr, Xte, ytr, yte = train_test_split(Xp, y_vec, test_size=0.3, random_state=SEED, stratify=y_vec)
    m = RandomForestClassifier(n_estimators=300, random_state=SEED, n_jobs=-1)
    m.fit(Xtr, ytr)
    yp = m.predict(Xte)
    return accuracy_score(yte, yp), f1_score(yte, yp, average="macro")

fg_acc, fg_f1 = eval_rf(X_fg_cmp, cmp_labels)
bio_acc, bio_f1 = eval_rf(X_bio_cmp, cmp_labels)

cmp_df = pd.DataFrame([
    {"backend": "fixed_gap", "runtime_s": t_fg_cmp, "rf_accuracy": fg_acc, "rf_macro_f1": fg_f1},
    {"backend": "biopython", "runtime_s": t_bio_cmp, "rf_accuracy": bio_acc, "rf_macro_f1": bio_f1},
])
display(cmp_df)
print(f"BioPython / fixed-gap runtime ratio: {t_bio_cmp / max(t_fg_cmp, 1e-9):.2f}x")

## Notes

- This notebook keeps the mirpy-native implementation compact and reproducible.
- `fixed_gap` is the default embedding backend in mirpy; `biopython` is included here as a semantic/quality comparison baseline.
- For large-scale runs, tune `n_prototypes`, class balancing limits, and DBSCAN parameters for your dataset and runtime budget.